In [2]:
%load_ext autoreload
%autoreload 2

import sentiments_utils as utils
import torch
import pandas as pd
from transformers import pipeline, set_seed
from datasets import load_dataset

2025-12-05 11:37:16.555807: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-05 11:37:17.532807: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-12-05 11:37:19.273533: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [3]:
# Dataset configuration
DATASET_PATH = "sentiment_datasets"
DATASET_SPLITS = {"train": f"{DATASET_PATH}/train.jsonl",
                  "validation": f"{DATASET_PATH}/val.jsonl",
                  "test": f"{DATASET_PATH}/test.jsonl"}
# Language configuration
SOURCE_COLUMN = "shp" 
TARGET_COLUMN = "spa"
LABEL_COLUMN = "sentiment"
NUMBER_LABELS = 3  # Positive, Negative, Neutral
# General configuration
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Training configuration
CONFIDENCE_THRESHOLD = 0.6
MAX_LENGTH = 128
SEED = 120
set_seed(SEED)

In [4]:
# Cargar datasets
full_dataset = load_dataset("json", data_files=DATASET_SPLITS)
train_dataset = full_dataset["train"]
validation_dataset = full_dataset["validation"]
test_dataset = full_dataset["test"]

# Separar 100 ejemplos del conjunto de pruebas para validación por expertos
expert_validation_dataset = test_dataset.filter(lambda example: example[SOURCE_COLUMN][0].isupper())  
expert_validation_dataset = expert_validation_dataset.filter(lambda example: len(example[SOURCE_COLUMN].strip()) < 70)
expert_validation_dataset = expert_validation_dataset.shuffle(seed=SEED).select(range(100))
# Usar el resto como conjunto de pruebas final
test_dataset = test_dataset.filter(lambda example: example not in expert_validation_dataset)

train_dataset, validation_dataset, test_dataset, expert_validation_dataset

(Dataset({
     features: ['spa', 'shp', 'sentiment', 'sentiment_confidence', 'sentiment_explanation'],
     num_rows: 11608
 }),
 Dataset({
     features: ['spa', 'shp', 'sentiment', 'sentiment_confidence', 'sentiment_explanation'],
     num_rows: 2902
 }),
 Dataset({
     features: ['spa', 'shp', 'sentiment', 'sentiment_confidence', 'sentiment_explanation'],
     num_rows: 896
 }),
 Dataset({
     features: ['spa', 'shp', 'sentiment', 'sentiment_confidence', 'sentiment_explanation'],
     num_rows: 100
 }))

# Modelo 1: XLM-Roberta-Base Shipibo

In [5]:
model_name_1 = "chechdop/xlm-roberta-sentiment-shp-tokenizer-vocab"
pipeline_1 = pipeline("sentiment-analysis", model=model_name_1, tokenizer=model_name_1, device=0 if DEVICE=="cuda" else -1)

Device set to use cuda:0


In [9]:
# Evaluar modelo con dataset de validacion
results_test_1 = utils.evaluate_dataset(pipeline_1, test_dataset, source_column=SOURCE_COLUMN, label_column=LABEL_COLUMN)
results_test_1

Evaluating: 100%|██████████| 56/56 [00:04<00:00, 12.61it/s]


{'accuracy': 0.5837053571428571,
 'balanced_accuracy': 0.5518576954929628,
 'precision': 0.5838562889715249,
 'recall': 0.5837053571428571,
 'f1': 0.5785544878783239,
 'confusion_matrix': array([[106,  98,  33],
        [ 51, 299,  76],
        [ 16,  99, 118]])}

In [7]:
predictions_test_1 = utils.predict_dataset_with_scores(pipeline_1, expert_validation_dataset, source_column=SOURCE_COLUMN, label_column=LABEL_COLUMN)
predictions_test_1_df = pd.DataFrame(predictions_test_1)
predictions_test_1_df.to_csv("expert_validation_predictions_model_1.csv", index=False)

Evaluating: 100%|██████████| 7/7 [00:00<00:00,  7.62it/s]


In [10]:
from huggingface_hub import notebook_login
notebook_login()

In [12]:
pipeline_1.push_to_hub("chechdop/xlm-roberta-sentiment-shp")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/chechdop/xlm-roberta-sentiment-shp/commit/6de54a2f4b9a9ba2af4da2a2bd398fe37370c373', commit_message='Upload TextClassificationPipeline', commit_description='', oid='6de54a2f4b9a9ba2af4da2a2bd398fe37370c373', pr_url=None, repo_url=RepoUrl('https://huggingface.co/chechdop/xlm-roberta-sentiment-shp', endpoint='https://huggingface.co', repo_type='model', repo_id='chechdop/xlm-roberta-sentiment-shp'), pr_revision=None, pr_num=None)

### Modelo 2: XLM-Roberta-Base v1

In [13]:
model_name_2 = "chechdop/xlm-roberta-sentiment-shp"
pipeline_2 = pipeline("sentiment-analysis", model=model_name_2, tokenizer=model_name_2, device=0 if DEVICE=="cuda" else -1)

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


In [ ]:
pipeline_2("Irake")

In [14]:
# Evaluar modelo con dataset de validacion
results_1 = utils.evaluate_dataset(pipeline_2, test_dataset, source_column=SOURCE_COLUMN, label_column=LABEL_COLUMN)
results_1

Evaluating: 100%|██████████| 56/56 [00:05<00:00, 10.97it/s]


{'accuracy': 0.5837053571428571,
 'balanced_accuracy': 0.5518576954929628,
 'precision': 0.5838562889715249,
 'recall': 0.5837053571428571,
 'f1': 0.5785544878783239,
 'confusion_matrix': array([[106,  98,  33],
        [ 51, 299,  76],
        [ 16,  99, 118]])}